# Ôn tập Buổi 06 - Cleaning, GroupBy và Pivot Table

        **Thời lượng gợi ý:** 60-75 phút  
        **Cách học:** trả lời câu hỏi trước khi chạy cell; sau mỗi ví dụ, tự nói thành lời *đầu vào - phép biến đổi - đầu ra*.

        ## Mục tiêu

        - Phát hiện duplicate/sentinel/outlier trước khi thay đổi.
- Dùng Split-Apply-Combine và named aggregation.
- Đọc pivot table cùng kích thước mẫu để tránh kết luận vội.

        > Notebook này là tài liệu ôn chủ động, không thay thế toàn bộ slide. Khi một câu tự kiểm tra chưa chắc, quay lại đúng mục tương ứng trong `slides/buoi6_python_datascience.pdf`.


In [ ]:
from pathlib import Path

HERE = Path.cwd().resolve()
ROOT = next(
    (p for p in (HERE, *HERE.parents) if (p / "datasets").exists() and (p / "slides").exists()),
    None,
)
assert ROOT is not None, "Hãy mở notebook từ bên trong repo Hoan-Data-Science-Course."
import numpy as np
import pandas as pd
DATA_DIR = ROOT / 'datasets' / 'buoi6'
print(f'Repo: {ROOT}')


## 0. Chẩn đoán nhanh - chưa chạy code

**1. `map()` gặp key không có trong dictionary thì sao?**

<details><summary>Kiểm tra đáp án</summary>

Thường trả `NaN`; phải kiểm tra sau ánh xạ.

</details>

**2. `agg()` và `transform()` khác nhau thế nào?**

<details><summary>Kiểm tra đáp án</summary>

`agg()` thu gọn mỗi nhóm; `transform()` trả kết quả cùng số dòng để gắn lại dữ liệu gốc.

</details>

**3. Có nên xóa mọi outlier?**

<details><summary>Kiểm tra đáp án</summary>

Không. Trước hết kiểm tra lỗi nhập liệu, đơn vị và bản chất hiện tượng.

</details>


## 1. Duplicate, sentinel và chuẩn hóa nhãn


In [ ]:
raw = pd.DataFrame({
    "id": [1, 2, 2, 3, 4],
    "city": [" HCM", "hcm", "hcm", "HN ", "unknown"],
    "score": [8, 7, 7, -999, 9],
})
audit = {"duplicate_rows": int(raw.duplicated().sum()), "sentinel_scores": int(raw["score"].eq(-999).sum())}
print(audit)
cleaned = raw.drop_duplicates().replace({"score": {-999: np.nan}}).copy()
cleaned["city"] = cleaned["city"].str.strip().str.upper().replace("UNKNOWN", pd.NA)
display(cleaned)
assert len(cleaned) == 4


## 2. `cut`, `qcut` và kiểm tra outlier


In [ ]:
ages = pd.Series([18, 20, 22, 25, 31, 37, 45, 61], name="age")
fixed_bins = pd.cut(ages, bins=[0, 24, 39, 59, np.inf], labels=["<=24", "25-39", "40-59", "60+"])
quantile_bins = pd.qcut(ages, q=4, duplicates="drop")

q1, q3 = ages.quantile([0.25, 0.75])
iqr = q3 - q1
bounds = (q1 - 1.5 * iqr, q3 + 1.5 * iqr)
outlier_mask = ~ages.between(*bounds)
print(pd.DataFrame({"age": ages, "fixed": fixed_bins, "quantile": quantile_bins, "outlier": outlier_mask}))
assert fixed_bins.notna().all()


## 3. GroupBy = Split -> Apply -> Combine


In [ ]:
titanic = pd.read_csv(DATA_DIR / "titanic.csv")
summary = (
    titanic.groupby(["class", "sex"], observed=True)
    .agg(passengers=("survived", "size"), survival_rate=("survived", "mean"), median_fare=("fare", "median"))
    .reset_index()
)
display(summary)
assert summary["passengers"].sum() == len(titanic)


## 4. `transform()` giữ nguyên số dòng

`fare_vs_class_median > 1` nghĩa là giá vé cao hơn median trong chính hạng vé của hành khách đó.


In [ ]:
class_median = titanic.groupby("class", observed=True)["fare"].transform("median")
titanic_enriched = titanic.assign(fare_vs_class_median=titanic["fare"] / class_median)
assert len(titanic_enriched) == len(titanic)
display(titanic_enriched[["class", "fare", "fare_vs_class_median"]].head())


## 5. Pivot table: hàng x cột và phải đọc cùng count


In [ ]:
survival = pd.pivot_table(
    titanic, values="survived", index="sex", columns="class", aggfunc="mean", observed=True
)
counts = pd.pivot_table(
    titanic, values="survived", index="sex", columns="class", aggfunc="size", observed=True
)
display(survival.style.format("{:.1%}"))
display(counts)
assert int(counts.to_numpy().sum()) == len(titanic)


## Bài tự luyện

        Tạo bảng theo `embark_town` và `class` gồm số hành khách, tuổi trung vị, giá vé trung vị. Sắp nhóm có nhiều hành khách nhất lên đầu.

        <details><summary>Gợi ý / đáp án tham khảo</summary>

        ```python
        answer = (
    titanic.groupby(["embark_town", "class"], observed=True)
    .agg(passengers=("survived", "size"), median_age=("age", "median"), median_fare=("fare", "median"))
    .sort_values("passengers", ascending=False)
)
        ```

        </details>


## Phiếu rời buổi

        Không nhìn lại notebook, hãy tự xác nhận:

        - [ ] Tôi audit trước khi drop/replace/cap dữ liệu.
- [ ] Tôi phân biệt `agg`, `transform`, `filter`, `apply`.
- [ ] Tôi đọc tỷ lệ cùng count trong GroupBy/Pivot.

        Nếu chưa đánh dấu được một mục, ghi lại **một ví dụ do chính bạn nghĩ ra** rồi chạy thử.
